# P18 — Aprender modelos visuales transferibles con supervisión de lenguaje natural

## 1. Título y paper

**Paper:** *Learning Transferable Visual Models From Natural Language Supervision*  
**Autoría:** Alec Radford, Jong Wook Kim, Chris Hallacy, Aditya Ramesh, Gabriel Goh, y otros (OpenAI)  
**Año y venue:** 2021 · arXiv:2103.00020 · ICML 2021  
**Nivel:** L3 · **Motor:** `clip`  
**Ficha completa:** [`P18_clip`](../../papers/foundational/P18_clip/README.md)

**Hito:** El texto se convierte en la etiqueta: un solo modelo clasifica categorías que nadie anotó, describiéndolas con palabras.

- [arXiv:2103.00020](https://arxiv.org/abs/2103.00020)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: La visión dependía de conjuntos etiquetados con categorías fijas; cambiar de tarea exigía volver a anotar y volver a entrenar.
2. Ejecutar una implementación mínima de la propuesta: Entrenar de forma contrastiva sobre 400 millones de pares (imagen, texto) de internet, alineando ambos espacios, y clasificar comparando la imagen con el texto de cada clase.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P04
- P05
- P08


## 4. Intuición

En vez de enseñar «esto es un gato» con una etiqueta numérica, se enseña con la frase que ya acompañaba a la foto en internet. El modelo aprende a acercar cada imagen a su texto y a alejarla de los textos de las demás.


## 5. Concepto mínimo

```text
InfoNCE simétrico sobre un lote de N pares (imagen_i, texto_i):

    logits_ij = cos(I_i, T_j) / τ
    L = ½·CE(logits, diagonal)_filas + ½·CE(logits, diagonal)_columnas
```

La diagonal son los pares correctos; **todo lo demás del lote son negativos**. Por eso el tamaño del lote es un hiperparámetro de primer orden: más lote, negativos más difíciles.


## 6. Código explicado

El motor entrena el contraste sobre cuatro pares y mide la matriz de similitud antes y después.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('clip', seed=7)['result']
print('conceptos:', r['conceptos'])
print('\nmatriz de similitud ANTES (filas=imagen, columnas=texto):')
for c, fila in zip(r['conceptos'], r['matriz_antes']):
    print(f'  {c:<7}', fila)
print('\nDESPUES:')
for c, fila in zip(r['conceptos'], r['matriz_despues']):
    print(f'  {c:<7}', fila)
show(r['diagonal_media'])
print('zero-shot:', r['zero_shot'])

## 7. Predicción antes de ejecutar

1. ¿Qué debe pasarle a la diagonal de la matriz? ¿Y a lo de fuera de la diagonal?
2. Con 4 conceptos, ¿cuántos negativos tiene cada positivo?
3. ¿Por qué esto permite clasificar sin haber entrenado un clasificador?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('clip', seed=semilla)['result']
    d = r['diagonal_media']
    print(f"semilla {semilla:>2} · diagonal {d['antes']:+.3f} → {d['despues']:+.3f} "
          f"· fuera {r['fuera_de_diagonal_media_despues']:+.3f} · zero-shot {r['zero_shot']}")

## 9. Salida interpretable

La diagonal sube y lo de fuera baja: los dos espacios quedan alineados. El zero-shot funciona porque clasificar se reduce a **preguntar qué frase se parece más a esta imagen** — y las frases se pueden escribir sobre la marcha, sin reentrenar nada.


## 10. Comentario pedagógico

Ojo con el nombre «zero-shot»: no significa que el modelo no haya visto gatos. Significa que no vio **este conjunto de etiquetas**. Con 400 millones de pares de internet, casi ninguna categoría común es realmente nueva, y el propio paper discute ese matiz.


## 11. Error o anti-patrón deliberado

Anti-patrón: evaluar zero-shot con las mismas plantillas de texto que se ajustaron mirando el conjunto de test.


In [ ]:
print('«a photo of a {clase}» vs «{clase}» vs «una foto de un {clase}, primer plano»')
print('La exactitud cambia varios puntos solo con la plantilla.')
print('Elegir la plantilla mirando el test convierte zero-shot en ajuste encubierto.')

## 12. Corrección

El protocolo honesto separa el conjunto donde se eligen las plantillas del conjunto donde se reporta:


In [ ]:
protocolo = {
    'plantillas_elegidas_en': 'conjunto de validación separado',
    'resultado_reportado_en': 'test, una sola vez',
    'se_reporta': ['plantilla exacta', 'número de plantillas probadas', 'varianza entre ellas'],
}
show(protocolo)

## 13. Desafío guiado

Comprueba que la matriz es simétrica en su papel: el negativo de una fila es el positivo de otra columna.


In [ ]:
r = run_paper_lab('clip', seed=7)['result']
m = r['matriz_despues']
n = len(m)
diag = [m[i][i] for i in range(n)]
fuera = [m[i][j] for i in range(n) for j in range(n) if i != j]
print('mínimo de la diagonal :', min(diag))
print('máximo fuera de ella  :', max(fuera))
print('¿separación perfecta? :', min(diag) > max(fuera))

## 14. Desafío autónomo

Con un modelo CLIP abierto y ejecutable localmente, construye 20 categorías propias y mide la exactitud zero-shot con tres plantillas distintas. Reporta media y varianza, y localiza una categoría donde falle sistemáticamente; explica por qué.


## 15. Evidencia de aprendizaje

Guarda las dos matrices de similitud, el resultado zero-shot y el protocolo de plantillas que usarías para que el número sea creíble.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P18_clip/README.md) · evaluación formal: [`assessments/papers/P18_clip.md`](../../assessments/papers/P18_clip.md)


## 16. Cierre

Imagen y texto ya viven en el mismo espacio. La pregunta siguiente no es de arquitectura sino de economía: dado un presupuesto de cómputo, ¿en qué conviene gastarlo?


## 17. Conexión con el siguiente hito

- modelos multimodales y de generación condicionada por texto

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
